In [56]:
import pandas as pd
import numpy as np
import navis
import os
import time
import math
import datetime
from tqdm import tqdm
import random
import re
import ast
import requests
from seatable_api import Base


# For flywire -> catmaid operations
import cloudvolume as cv
import caveclient

from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
import matplotlib.colors as mcl

In [4]:
# which datastack? 
datastack = 'PB1' # Options: FBEB, PB1... # note PB2 not actually used

client = caveclient.CAVEclient(server_address='https://global.connectomics.braininbrain.org')


# NOTE PB SYNAPSES ARE NOT ONLINE
if datastack == 'PB1':
    client = caveclient.CAVEclient(datastack_name='megalopta_pb1_datastack')
    auth = client.auth
    cg = client.chunkedgraph
    print('client set for PB data')

elif datastack == 'PB2':
    client = caveclient.CAVEclient(datastack_name='megalopta_pb2_datastack')
    auth = client.auth
    cg = client.chunkedgraph
    print('client set for PB2 data')

elif datastack == 'FBEB':
    client = caveclient.CAVEclient(datastack_name='megalopta_fb_eb_datastack')
    auth = client.auth
    cg = client.chunkedgraph
    print('client set for EB data')

elif datastack == 'NO':
    client = caveclient.CAVEclient(datastack_name='megalopta_no_r_datastack')
    auth = client.auth
    cg = client.chunkedgraph
    print('client set for noduli data')


client set for PB data


### import neuron nametable (names, root ids, catmaid skid) Google Sheets .csv

***NOTE: all root ids must be up to date. See update_googlesheet_rootid notebook in /utils***

In [5]:
if datastack == 'FBEB':
    epg = pd.read_csv('./updated_google_sheets/Megalopta_EB_preprint_FINAL - FBEB_EPG.csv', dtype={'Root ID': str})
    pen = pd.read_csv('./updated_google_sheets/Megalopta_EB_preprint_FINAL - FBEB_PEN.csv', dtype={'Root ID': str})
    er = pd.read_csv('./updated_google_sheets/Megalopta_EB_preprint_FINAL - FBEB_ER.csv', dtype={'Root ID': str})
    #er = er[er['Completed']==True]
    nametable = pd.concat((epg, pen, er))
elif datastack == 'PB1':
    epg = pd.read_csv('./updated_google_sheets/Megalopta_PB_preprint_FINAL - PB1_EPG.csv', dtype={'Root ID': str})
    pen = pd.read_csv('./updated_google_sheets/Megalopta_PB_preprint_FINAL - PB1_PEN.csv', dtype={'Root ID': str}) 
    d7 = pd.read_csv('./updated_google_sheets/Megalopta_PB_preprint_FINAL - PB1_delta7.csv', dtype={'Root ID': str})
    nametable = pd.concat((epg, pen, d7))
elif datastack == 'PB2':
    d7 = pd.read_csv('./updated_google_sheets/Megalopta_PB2_preprint_FINAL - PB2_delta7.csv', dtype={'Root ID': str})
    epg = pd.read_csv('./updated_google_sheets/Megalopta_PB2_preprint_FINAL - PB2_EPG.csv', dtype={'Root ID': str})
    nametable = pd.concat((d7, epg))
elif datastack == 'NO':
    lno = pd.read_csv('./updated_google_sheets/Megalopta_NOr neuron_CAVE progress - NOr_LNO.csv', dtype={'Root ID': str})
    pen = pd.read_csv('./updated_google_sheets/Megalopta_NOr neuron_CAVE progress - NOr_PEN.csv', dtype={'Root ID': str})
    nametable = pd.concat((lno, pen))

if datastack == 'NO':
    nametable = nametable[['Catmaid name', 'Latest segment IDs', 'CATMAID skid']].rename(columns={'Catmaid name':'name', 'Latest segment IDs':'root_ids', 'CATMAID skid':'skid'})
else:
    nametable = nametable[['Catmaid name', 'Root ID', 'CATMAID skid']].rename(columns={'Catmaid name':'name', 'Root ID':'root_ids', 'CATMAID skid':'skid'})

# TEMPORARY - THIS NEURON HASN'T BEEN PROOFREAD IN PB2
if datastack == 'PB2':
    nametable = nametable[nametable['name'] != 'delta7_R_L3R6_79852']

nametable['skid'] = nametable['skid'].fillna(0).astype(float).astype(int)

# make sure to fill na
nametable['root_ids'] = nametable['root_ids'].fillna('')

# convert each cell in 'root_ids' from a comma-separated string to a list of integers, ignoring empty strings
nametable['root_ids'] = nametable['root_ids'].apply(lambda x: [int(seg) for seg in x.split(',') if seg.strip()])

# remove empty values
nametable = nametable[nametable['root_ids'].apply(lambda x: len(x) > 0)].copy()

if datastack == 'FBEB':
    nametable = nametable[~nametable['name'].str.contains("L4|L5|L6|L7|L8|L9|R4|R5|R6|R7|PEG_L3|PFx|unknown", regex=True)]
elif datastack == 'PB1':
    nametable = nametable[~nametable['name'].str.contains("LX_L7", regex=True)]

# Convert each element of root_ids from a list to multiple rows
nametable_exploded = nametable.explode('root_ids')

# Remove rows with NaNs in root_ids
nametable_exploded = nametable_exploded.dropna(subset=['root_ids'])

# Ensure root_ids is of a numeric type
nametable_exploded['root_ids'] = nametable_exploded['root_ids'].astype('int64')

# Deduplicate the rows so that each root_id appears only once
nametable_exploded = nametable_exploded.drop_duplicates(subset='root_ids')

nametable_exploded

,name,root_ids,skid
0,EPG_L2_55944,576460752521344750,55944
1,EPG_L2_55952,576460752510496133,55952
2,EPG_L3_54939,576460752494001689,54939
2,EPG_L3_54939,576460752504703640,54939
3,EPG_L3_57518,576460752372935125,57518
...,...,...,...
38,delta7_R_L2L10R7_83722,576460752608580253,83722
39,delta7_R_L2L10R7_84178,576460752490172734,84178
40,delta7_R_L2L10R7_84473,576460752543874747,84473
40,delta7_R_L2L10R7_84473,576460752545051043,84473


# methods for updating synapse table

### using older version of synapse table (syntable) .csv file

***IMPORTANT: if using .csv ensure the .csv has all synapses! For instance, a syntable might have been created using the cave client to download synapses for a table of neurons. In this case, only the synapses associated with those neurons will have been included, not every single synapse***

'''
method to update synapse table using local .csv file
1. update neuron root ids using Google Sheet API (backbone coord -> supervox -> get root; see update_googlesheet_rootid notebook)
2. using an old synapse table: update root ids: supervoxel -> get latest root 
3. merge updated syntable to Google Sheets nametable using common root_ids (make sure root_ids are updated for Google Sheet nametable)
'''

In [20]:
'''
method to update synapse table using local .csv file
1. for syntablle update root ids: supervoxel -> get latest root 
2. merge synatble to Google Sheets nametable using common root_ids (make sure root_ids are updated for Google Sheet nametable)
'''
if datastack == 'FBEB': 
    synapse_df = pd.read_csv('../syntables/table_csv_files/megalopta_EB_syntable_raw_06-Dec-25.csv')
elif datastack == 'PB1': 
    synapse_df = pd.read_csv('../syntables/table_csv_files/old_v2/PB_all_syns_feb9_20_wscores_with-roots_with-sk_24-10-30.csv')
elif datastack == 'PB2': 
    synapse_df = pd.read_csv('../syntables/table_csv_files/megalopta_pb2_with-roots_with-sk_26-03-28.csv')

for c in tqdm([c for c in synapse_df.columns if 'coords' in c or 'vox' in c]): 
    synapse_df[c] = synapse_df[c].apply(lambda s: tuple(map(int, s.strip('()').split(','))))


100%|████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:04<00:00,  1.43it/s]


In [106]:
synapse_df.columns

Index(['Unnamed: 0.2', 'index', 'Unnamed: 0.1', 'Unnamed: 0', 'pre_vox_coord',
       'post_vox_coord', 'pre_chunk_coords', 'post_chunk_coords',
       'local_pre_vox', 'local_post_vox', 'pre_chunk_ids', 'post_chunk_ids',
       'pre_L1_ids', 'post_L1_ids', 'pre_root_ids', 'post_root_ids',
       'pre_neuron_name', 'pre_skid', 'pre_n_children', 'pre_children_layer',
       'post_neuron_name', 'post_skid', 'post_n_children',
       'post_children_layer'],
      dtype='object')

In [21]:
# check duplicates
dups = synapse_df[
    synapse_df.duplicated(subset=["pre_vox_coord", "post_vox_coord"], keep=False)
].sort_values(by=["pre_vox_coord", "post_vox_coord"])

dups

,Unnamed: 0.2,index,Unnamed: 0.1,Unnamed: 0,pre_vox_coord,post_vox_coord,pre_chunk_coords,post_chunk_coords,local_pre_vox,local_post_vox,...,pre_root_ids,post_root_ids,pre_neuron_name,pre_skid,pre_n_children,pre_children_layer,post_neuron_name,post_skid,post_n_children,post_children_layer
262,262,262,262,262,"(29359, 5645, 1926)","(29344, 5669, 1925)","(0, 3, 10)","(0, 4, 10)","(148, 485, 49)","(133, 9, 48)",...,576460752340423325,576460752449481928,PBneuron_different6,61484.0,1.0,2.0,EXR_PB_TL_SH,62965.0,1.0,2.0
263,263,263,263,263,"(29359, 5645, 1926)","(29344, 5669, 1925)","(0, 3, 10)","(0, 4, 10)","(148, 485, 49)","(133, 9, 48)",...,576460752340423325,576460752449481928,PBneuron_different6,61484.0,1.0,2.0,EXR_PB_TL_SH,62965.0,1.0,2.0
259,259,259,259,259,"(29359, 5645, 1926)","(29383, 5666, 1926)","(0, 3, 10)","(0, 4, 10)","(148, 485, 49)","(172, 6, 49)",...,576460752340423325,576460752451936200,PBneuron_different6,61484.0,1.0,2.0,NaN,NaN,NaN,NaN
260,260,260,260,260,"(29359, 5645, 1926)","(29383, 5666, 1926)","(0, 3, 10)","(0, 4, 10)","(148, 485, 49)","(172, 6, 49)",...,576460752340423325,576460752451936200,PBneuron_different6,61484.0,1.0,2.0,NaN,NaN,NaN,NaN
3945,3945,3945,3945,3945,"(29428, 7699, 1901)","(29428, 7674, 1900)","(0, 8, 10)","(0, 8, 10)","(217, 39, 24)","(217, 14, 23)",...,576460752313729249,576460752338793944,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
469443,470219,470219,470219,470219,"(51463, 5065, 1755)","(51485, 5051, 1755)","(44, 2, 8)","(44, 2, 8)","(252, 405, 78)","(274, 391, 78)",...,576460752425218090,576460752408891057,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
676275,677315,677315,677315,677315,"(52067, 8813, 1925)","(52087, 8832, 1924)","(45, 10, 10)","(45, 10, 10)","(356, 153, 48)","(376, 172, 47)",...,576460752519473593,576460752437174889,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
676276,677316,677316,677316,677316,"(52067, 8813, 1925)","(52087, 8832, 1924)","(45, 10, 10)","(45, 10, 10)","(356, 153, 48)","(376, 172, 47)",...,576460752519473593,576460752437174889,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
676277,677317,677317,677317,677317,"(52067, 8813, 1925)","(52088, 8826, 1925)","(45, 10, 10)","(45, 10, 10)","(356, 153, 48)","(377, 166, 48)",...,576460752519473593,576460752437271145,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
# Identify groups that have duplicates
dup_mask = synapse_df.duplicated(subset=["pre_vox_coord", "post_vox_coord"], keep=False)

# For each duplicated group, drop exactly one row
to_drop = synapse_df[dup_mask].groupby(["pre_vox_coord", "post_vox_coord"]).head(1).index

# New DF with one instance removed
synapse_df = synapse_df.drop(index=to_drop)

In [23]:
'''reset neuron names / types'''

synapse_df[['pre_neuron_name',
            'post_neuron_name',
            'pre_skid',
            'post_skid']] = np.nan

In [24]:
'''
method to update synapse table using local .csv file
1. for synapse_df update root ids: supervoxel -> get latest root 
2. merge synatble to Google Sheets nametable using common root_ids (make sure root_ids are updated for Google Sheet nametable)
'''

def coerce_sv_int(x):
    """Parse supervoxel IDs from messy cells: lists/tuples/strings, commas, sci-notation."""
    if pd.isna(x):
        return None
    if isinstance(x, (int, np.integer)):
        return int(x)
    if isinstance(x, (list, tuple, np.ndarray)):
        # if cell holds multiple SVs, take the first (or change to your policy)
        if len(x) == 0: 
            return None
        return coerce_sv_int(x[0])

    s = str(x).strip()
    # strip brackets/braces/parentheses
    s = s.strip("[](){}")
    # take the first token if comma/space-separated
    token = re.split(r"[,\s]+", s)[0]
    try:
        return int(token)
    except Exception:
        try:
            # handle things like '1.23e+11' or '123.0'
            return int(np.int64(float(token)))
        except Exception:
            return None

# Re-parse columns
pre_ids_raw  = synapse_df['pre_L1_ids']
post_ids_raw = synapse_df['post_L1_ids']

pre_ids  = pre_ids_raw.map(coerce_sv_int)
post_ids = post_ids_raw.map(coerce_sv_int)

print("Parse diagnostics:")
print(" pre valid:", pre_ids.notna().sum(), " / ", len(pre_ids))
print(" post valid:", post_ids.notna().sum(), " / ", len(post_ids))

# Unique IDs to resolve
all_ids = pd.Index(pd.concat([pre_ids, post_ids], ignore_index=True)).dropna().astype(np.int64).unique().tolist()
print(f"Unique supervoxel IDs to resolve: {len(all_ids):,}")

# Batch resolve
batch_size = 20_000
sv2root = {}
for i in tqdm(range(0, len(all_ids), batch_size), desc="Resolving roots", unit="batch"):
    batch = all_ids[i:i+batch_size]
    roots = cg.get_roots(batch)     # returns list aligned with batch
    for sv, root in zip(batch, roots):
        # Some deployments return [root] or np.array([root])
        if isinstance(root, (list, tuple, np.ndarray)):
            root = root[0] if len(root) else None
        sv2root[int(sv)] = int(root) if root is not None else None

# Map back
synapse_df['pre_root_ids_updated']  = pre_ids.map(sv2root)
synapse_df['post_root_ids_updated'] = post_ids.map(sv2root)

print("Missing pre roots: ", synapse_df['pre_root_ids_updated'].isna().sum())
print("Missing post roots:", synapse_df['post_root_ids_updated'].isna().sum())

# inspect some misses to confirm parsing vs. lookup issues
miss_pre = synapse_df.loc[synapse_df['pre_root_ids_updated'].isna(), ['pre_L1_ids']].head(10)
miss_post = synapse_df.loc[synapse_df['post_root_ids_updated'].isna(), ['post_L1_ids']].head(10)
print("Examples of pre misses:\n", miss_pre)
print("Examples of post misses:\n", miss_post)

Parse diagnostics:
 pre valid: 673741  /  673741
 post valid: 673741  /  673741
Unique supervoxel IDs to resolve: 1,171,888


Resolving roots: 100%|██████████████████████████████████████████████████████████████| 59/59 [02:30<00:00,  2.54s/batch]


Missing pre roots:  0
Missing post roots: 0
Examples of pre misses:
 Empty DataFrame
Columns: [pre_L1_ids]
Index: []
Examples of post misses:
 Empty DataFrame
Columns: [post_L1_ids]
Index: []


In [116]:
synapse_df.columns

Index(['Unnamed: 0.2', 'index', 'Unnamed: 0.1', 'Unnamed: 0', 'pre_vox_coord',
       'post_vox_coord', 'pre_chunk_coords', 'post_chunk_coords',
       'local_pre_vox', 'local_post_vox', 'pre_chunk_ids', 'post_chunk_ids',
       'pre_L1_ids', 'post_L1_ids', 'pre_root_ids', 'post_root_ids',
       'pre_neuron_name', 'pre_skid', 'pre_n_children', 'pre_children_layer',
       'post_neuron_name', 'post_skid', 'post_n_children',
       'post_children_layer', 'pre_root_ids_updated', 'post_root_ids_updated'],
      dtype='object')

In [25]:
#syntable = synapse_df[~(synapse_df['pre_root_ids_updated']==0)|(synapse_df['post_root_ids_updated']==0)] 
# filter values w/ no supervoxel
syntable = synapse_df.drop(columns=['pre_root_ids', 'post_root_ids'])
syntable = syntable.rename(columns={'pre_root_ids_updated': 'pre_pt_root_id', 'post_root_ids_updated': 'post_pt_root_id'})

***IMPORTANT: ensure columns are of same type. Otherwise will merge and, even if fails, will error silently***

In [26]:
# 0) Clean any pre-existing columns that will be re-created by the merges
cols_to_drop = [c for c in ['pre_neuron_name','post_neuron_name','pre_skid','post_skid'] 
                if c in syntable.columns]
syntable = syntable.drop(columns=cols_to_drop)

# ensure data types match prior to merge
syntable["pre_pt_root_id"] = pd.to_numeric(syntable["pre_pt_root_id"], errors="coerce").astype("Int64")
syntable["post_pt_root_id"] = pd.to_numeric(syntable["post_pt_root_id"], errors="coerce").astype("Int64")
nametable_exploded["root_ids"] = pd.to_numeric(nametable_exploded["root_ids"], errors="coerce").astype("Int64")

# 1) First merge: pre side
syntable_merge = syntable.merge(
    nametable_exploded[['root_ids','name','skid']],
    left_on='pre_pt_root_id', right_on='root_ids',
    how='left'
).rename(columns={'name':'pre_neuron_name','skid':'pre_skid'}).drop(columns=['root_ids'])

# 2) Second merge: post side
syntable_merge = syntable_merge.merge(
    nametable_exploded[['root_ids','name','skid']],
    left_on='post_pt_root_id', right_on='root_ids',
    how='left'
).rename(columns={'name':'post_neuron_name','skid':'post_skid'}).drop(columns=['root_ids'])

# (Optional) sanity check: ensure no duplicate pre/post_skid columns remain
assert (syntable_merge.columns == 'pre_skid').sum()  == 1
assert (syntable_merge.columns == 'post_skid').sum() == 1

# 3) Remove autapses: only if BOTH skids are present and equal
pre_s = syntable_merge['pre_skid'].astype('Int64')   # nullable int
post_s = syntable_merge['post_skid'].astype('Int64')

is_autapse = pre_s.notna() & post_s.notna() & (pre_s == post_s)
syntable_filt = syntable_merge[~is_autapse]
syntable_filt

,Unnamed: 0.2,index,Unnamed: 0.1,Unnamed: 0,pre_vox_coord,post_vox_coord,pre_chunk_coords,post_chunk_coords,local_pre_vox,local_post_vox,...,pre_n_children,pre_children_layer,post_n_children,post_children_layer,pre_pt_root_id,post_pt_root_id,pre_neuron_name,pre_skid,post_neuron_name,post_skid
0,0,0,0,0,"(29539, 4469, 1872)","(29526, 4495, 1872)","(0, 1, 9)","(0, 1, 9)","(328, 309, 95)","(315, 335, 95)",...,NaN,NaN,NaN,NaN,576460752323320962,576460752334496438,NaN,NaN,NaN,NaN
1,1,1,1,1,"(29363, 4580, 1873)","(29359, 4601, 1873)","(0, 1, 9)","(0, 1, 9)","(152, 420, 96)","(148, 441, 96)",...,NaN,NaN,NaN,NaN,576460752352521639,576460752352475815,NaN,NaN,NaN,NaN
2,2,2,2,2,"(29386, 4596, 1872)","(29386, 4617, 1873)","(0, 1, 9)","(0, 1, 9)","(175, 436, 95)","(175, 457, 96)",...,NaN,NaN,NaN,NaN,576460752352458151,576460752323350658,NaN,NaN,NaN,NaN
3,3,3,3,3,"(29338, 4644, 1894)","(29329, 4637, 1894)","(0, 1, 10)","(0, 1, 10)","(127, 484, 17)","(118, 477, 17)",...,NaN,NaN,NaN,NaN,576460752312516286,576460752310445388,NaN,NaN,NaN,NaN
4,4,4,4,4,"(29539, 4803, 1917)","(29535, 4791, 1918)","(0, 2, 10)","(0, 2, 10)","(328, 143, 40)","(324, 131, 41)",...,NaN,NaN,NaN,NaN,576460752335006621,576460752339644573,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
673736,677438,677438,677438,677438,"(51885, 9881, 1965)","(51903, 9893, 1965)","(45, 12, 10)","(45, 12, 10)","(174, 221, 88)","(192, 233, 88)",...,NaN,NaN,NaN,NaN,576460752461930249,576460752472278526,NaN,NaN,NaN,NaN
673737,677439,677439,677439,677439,"(51760, 9944, 1969)","(51786, 9941, 1969)","(45, 12, 10)","(45, 12, 10)","(49, 284, 92)","(75, 281, 92)",...,NaN,NaN,NaN,NaN,576460752462291721,576460752472278526,NaN,NaN,NaN,NaN
673738,677440,677440,677440,677440,"(51957, 10753, 1947)","(51971, 10743, 1946)","(45, 14, 10)","(45, 14, 10)","(246, 93, 70)","(260, 83, 69)",...,NaN,NaN,NaN,NaN,576460752343854831,576460752340407053,NaN,NaN,NaN,NaN
673739,677441,677441,677441,677441,"(51717, 10195, 1949)","(51715, 10173, 1949)","(45, 13, 10)","(45, 13, 10)","(6, 35, 72)","(4, 13, 72)",...,NaN,NaN,NaN,NaN,576460752556436083,576460752556544115,NaN,NaN,NaN,NaN


### live query

In [63]:
client.materialize.get_timestamp()
#datetime.datetime(2025, 8, 2, 3, 10, 1, 610683, tzinfo=datetime.timezone.utc)

datetime.datetime(2026, 4, 7, 3, 10, 2, 58115, tzinfo=datetime.timezone.utc)

In [52]:
# LIVE
# get presynaptic synapses 
# obtain latest m. version
synapse_table = client.info.get_datastack_info()['synapse_table']
    
neuron_ids = nametable_exploded['root_ids'].to_list()

pre_syn = client.materialize.live_query(
    synapse_table,
    datetime.datetime.now(datetime.timezone.utc),
    filter_in_dict={'pre_pt_root_id': neuron_ids},
    select_columns=['id', 'pre_pt_root_id', 'pre_pt_position', 'post_pt_root_id', 'post_pt_position']
)

# get postsynaptic synapses 
post_syn = client.materialize.live_query(
    synapse_table,
    datetime.datetime.now(datetime.timezone.utc),
    filter_in_dict={'post_pt_root_id': neuron_ids},
    select_columns=['id', 'pre_pt_root_id', 'pre_pt_position', 'post_pt_root_id', 'post_pt_position']
)

syntable = pd.concat([pre_syn, post_syn]).copy()

# convert lists/arrays to strings so they can be hashed
syntable['pre_pt_position_str'] = syntable['pre_pt_position'].apply(lambda x: str(x))
syntable['post_pt_position_str'] = syntable['post_pt_position'].apply(lambda x: str(x))

# drop duplicates based on stringified positions
syntable = syntable.drop_duplicates(
    subset=['pre_pt_position_str', 'post_pt_position_str']
).reset_index(drop=True)

# match pre_pt_root_id to neuron names
syntable_merge = syntable.merge(
    right=nametable_exploded[['root_ids', 'name', 'skid']],
    right_on='root_ids',
    left_on='pre_pt_root_id',
    how='left'  # <-- this keeps all rows from syntable
).rename(columns={'name': 'pre_neuron_name', 'skid': 'pre_skid'})

# drop redundant 'root_ids' column from the first merge to avoid column name clash
syntable_merge = syntable_merge.drop(columns=['root_ids'])

# match post_pt_root_id to neuron names
syntable_merge = syntable_merge.merge(
    right=nametable_exploded[['root_ids', 'name', 'skid']],
    right_on='root_ids',
    left_on='post_pt_root_id',
    how='left'  # <-- again, keep all rows
).rename(columns={'name': 'post_neuron_name', 'skid': 'post_skid'})

syntable_merge = syntable_merge.drop(columns=['root_ids'])  
syntable_filt = syntable_merge[syntable_merge['pre_skid'] != syntable_merge['post_skid']] #remove autapses

syntable_filt

[2026-04-08 13:05:58] [WARNING] Query was executed using streaming via CSV, which can mangle types. Please upgrade to caveclient > 8.0.0 to avoid type mangling. Because you may have been affected by mangled types, this change is breaking, but it should provide an improved experience.
[2026-04-08 13:06:17] [INFO] pre_pt_supervoxel_id has 0 to update
[2026-04-08 13:06:17] [INFO] post_pt_supervoxel_id has 62 to update
[2026-04-08 13:06:17] [INFO] num zero svids: 0
[2026-04-08 13:06:17] [INFO] all_svids dtype int64
[2026-04-08 13:06:17] [INFO] all_svid_lengths [0, 62]
[2026-04-08 13:06:21] [WARNING] Query was executed using streaming via CSV, which can mangle types. Please upgrade to caveclient > 8.0.0 to avoid type mangling. Because you may have been affected by mangled types, this change is breaking, but it should provide an improved experience.
[2026-04-08 13:06:25] [INFO] pre_pt_supervoxel_id has 151 to update
[2026-04-08 13:06:25] [INFO] post_pt_supervoxel_id has 0 to update
[2026-04-

,id,pre_pt_supervoxel_id,pre_pt_root_id,post_pt_supervoxel_id,post_pt_root_id,pre_pt_position,post_pt_position,pre_pt_position_str,post_pt_position_str,pre_neuron_name,pre_skid,post_neuron_name,post_skid
0,2176,77759936217496846,576460752438319492,77759936217497446,576460752482567425,"[32113, 5926, 1814]","[32128, 5943, 1816]",[32113 5926 1814],[32128 5943 1816],delta7_L5R4_84466,84466.0,PEG_L2_55930,55930.0
1,6723,76669220682742017,576460752583772701,76669220682740645,576460752429928642,"[31517, 6948, 1876]","[31492, 6936, 1875]",[31517 6948 1876],[31492 6936 1875],delta7_R_L2L10R7_42181,42181.0,NaN,NaN
2,6724,76669220682739587,576460752583772701,76669220682739560,576460752429785794,"[31507, 6946, 1874]","[31482, 6959, 1874]",[31507 6946 1874],[31482 6959 1874],delta7_R_L2L10R7_42181,42181.0,NaN,NaN
3,6732,76669495560636610,576460752583772701,76669495560637629,576460752331804956,"[31525, 6951, 1878]","[31535, 6931, 1879]",[31525 6951 1878],[31535 6931 1879],delta7_R_L2L10R7_42181,42181.0,NaN,NaN
4,6746,76669495560641737,576460752583772701,76669495560641690,576460752332124700,"[31487, 7001, 1882]","[31462, 6982, 1882]",[31487 7001 1882],[31462 6982 1882],delta7_R_L2L10R7_42181,42181.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
300557,674294,116181545416496681,576460752409742888,116181545416497832,576460752453725753,"[49189, 9713, 1958]","[49187, 9735, 1959]",[49189 9713 1958],[49187 9735 1959],NaN,NaN,EPG_L6_57712,57712.0
300558,674236,116163953230444691,576460752304046438,116163953230445829,576460752453725753,"[48924, 9643, 1950]","[48935, 9637, 1951]",[48924 9643 1950],[48935 9637 1951],NaN,NaN,EPG_L6_57712,57712.0
300559,674248,116181545416494746,576460752471766306,116181545416494617,576460752453725753,"[49036, 9712, 1956]","[49042, 9694, 1956]",[49036 9712 1956],[49042 9694 1956],NaN,NaN,EPG_L6_57712,57712.0
300560,674485,121635123090239416,576460752531070332,121635123090237606,576460752438319492,"[51632, 5115, 1974]","[51654, 5126, 1972]",[51632 5115 1974],[51654 5126 1972],NaN,NaN,delta7_L5R4_84466,84466.0


### query latest version

In [23]:
synapse_table = client.info.get_datastack_info()['synapse_table']
print(synapse_table)

synapses_v2


In [26]:
# obtain latest m. version
synapse_table = client.info.get_datastack_info()['synapse_table']
print(f'synapse table: synapse_table')
    
neuron_ids = nametable_exploded['root_ids'].to_list()


pre_syn = client.materialize.query_table(
    synapse_table,
    filter_in_dict={'pre_pt_root_id': neuron_ids},
    select_columns=['id', 'pre_pt_root_id', 'pre_pt_position', 'post_pt_root_id', 'post_pt_position']
)
post_syn = client.materialize.query_table(
    synapse_table,
    filter_in_dict={'post_pt_root_id': neuron_ids},
    select_columns=['id', 'pre_pt_root_id', 'pre_pt_position', 'post_pt_root_id', 'post_pt_position']
)

syntable = pd.concat([pre_syn, post_syn]).copy()

# convert lists/arrays to strings so they can be hashed
syntable['pre_pt_position_str'] = syntable['pre_pt_position'].apply(lambda x: str(x))
syntable['post_pt_position_str'] = syntable['post_pt_position'].apply(lambda x: str(x))

# drop duplicates based on stringified positions
syntable = syntable.drop_duplicates(
    subset=['pre_pt_position_str', 'post_pt_position_str']
).reset_index(drop=True)

# ensure data types match prior to merge
syntable["pre_pt_root_id"] = pd.to_numeric(syntable["pre_pt_root_id"], errors="coerce").astype("Int64")
syntable["post_pt_root_id"] = pd.to_numeric(syntable["post_pt_root_id"], errors="coerce").astype("Int64")
nametable_exploded["root_ids"] = pd.to_numeric(nametable_exploded["root_ids"], errors="coerce").astype("Int64")

# match pre_pt_root_id to neuron names
syntable_merge = syntable.merge(
    right=nametable_exploded[['root_ids', 'name', 'skid']],
    right_on='root_ids',
    left_on='pre_pt_root_id',
    how='left'  # <-- this keeps all rows from syntable
).rename(columns={'name': 'pre_neuron_name', 'skid': 'pre_skid'})

# drop redundant 'root_ids' column from the first merge to avoid column name clash
syntable_merge = syntable_merge.drop(columns=['root_ids'])

# match post_pt_root_id to neuron names
syntable_merge = syntable_merge.merge(
    right=nametable_exploded[['root_ids', 'name', 'skid']],
    right_on='root_ids',
    left_on='post_pt_root_id',
    how='left'  # <-- again, keep all rows
).rename(columns={'name': 'post_neuron_name', 'skid': 'post_skid'})

syntable_merge = syntable_merge.drop(columns=['root_ids'])

syntable_merge["pre_skid"] = pd.to_numeric(syntable_merge["pre_skid"], errors="coerce").astype("Int64")
syntable_merge["post_skid"] = pd.to_numeric(syntable_merge["post_skid"], errors="coerce").astype("Int64")

syntable_filt = syntable_merge[syntable_merge['pre_skid'] != syntable_merge['post_skid']] #remove autapses

syntable_filt

synapse table: synapse_table
[2026-04-18 18:32:18] [WARNING] Query was executed using streaming via CSV, which can mangle types. Please upgrade to caveclient > 8.0.0 to avoid type mangling. Because you may have been affected by mangled types, this change is breaking, but it should provide an improved experience.
[2026-04-18 18:32:23] [WARNING] Query was executed using streaming via CSV, which can mangle types. Please upgrade to caveclient > 8.0.0 to avoid type mangling. Because you may have been affected by mangled types, this change is breaking, but it should provide an improved experience.


,id,pre_pt_supervoxel_id,pre_pt_root_id,post_pt_supervoxel_id,post_pt_root_id,pre_pt_position,post_pt_position,pre_pt_position_str,post_pt_position_str,pre_neuron_name,pre_skid,post_neuron_name,post_skid
1,2230,78903153432539010,576460752490172734,78903153432538959,576460752481686074,"[32380, 6339, 1738]","[32362, 6316, 1738]",[32380 6339 1738],[32362 6316 1738],delta7_R_L2L10R7_84178,84178,delta7_L_L6R3_84617,84617
22,7959,77795395467509924,576460752583772701,77795395467509864,576460752438319492,"[32202, 6706, 1901]","[32184, 6696, 1901]",[32202 6706 1901],[32184 6696 1901],delta7_R_L2L10R7_42181,42181,delta7_L5R4_84466,84466
66,24194,82297345827516275,576460752456808411,82297345827516373,576460752543834043,"[33979, 6831, 1373]","[33956, 6854, 1373]",[33979 6831 1373],[33956 6854 1373],delta7_R_L3R6_84354,84354,delta7_R_L4R5_110806,110806
83,25075,80046645525440861,576460752490172734,80046645525442210,576460752507193318,"[32748, 6731, 1751]","[32746, 6710, 1752]",[32748 6731 1751],[32746 6710 1752],delta7_R_L2L10R7_84178,84178,PEN_b_L2_112687,112687
90,25293,78921020496480778,576460752490172734,78921020496480896,576460752482567425,"[32686, 6720, 1828]","[32700, 6746, 1828]",[32686 6720 1828],[32700 6746 1828],delta7_R_L2L10R7_84178,84178,PEG_L2_55930,55930
...,...,...,...,...,...,...,...,...,...,...,...,...,...
259816,658558,118362426730204037,576460752541325943,118362426730203973,576460752461902026,"[50027, 8095, 1760]","[50051, 8091, 1760]",[50027 8095 1760],[50051 8091 1760],EPG_L6_60458,60458,PEN_b_L6_34205,34205
259825,658769,119506193700977758,576460752532127661,119506193700976348,576460752461902026,"[50305, 8271, 1860]","[50318, 8286, 1859]",[50305 8271 1860],[50318 8286 1859],delta7_L6R3_80994,80994,PEN_b_L6_34205,34205
259837,659135,118380018916194773,576460752612086749,118380018916194765,576460752461902026,"[50100, 8518, 1724]","[50085, 8518, 1724]",[50100 8518 1724],[50085 8518 1724],delta7_L6R3_82632,82632,PEN_b_L6_34205,34205
259857,670939,113859376858633850,576460752546883616,113859376858635034,576460752550908713,"[47786, 8133, 1961]","[47769, 8153, 1962]",[47786 8133 1961],[47769 8153 1962],EPG_L6_59381,59381,delta7_R_L2L10R7_83213,83213


### verify and export

In [29]:
# check some root id
syntable_filt[syntable_filt['pre_pt_root_id']==576460752504703640] # EPG_L3 PB datastack

,Unnamed: 0.2,index,Unnamed: 0.1,Unnamed: 0,pre_vox_coord,post_vox_coord,pre_chunk_coords,post_chunk_coords,local_pre_vox,local_post_vox,...,pre_n_children,pre_children_layer,post_n_children,post_children_layer,pre_pt_root_id,post_pt_root_id,pre_neuron_name,pre_skid,post_neuron_name,post_skid
72687,73114,73114,73114,73114,"(34371, 8914, 1574)","(34363, 8933, 1575)","(10, 10, 6)","(10, 10, 6)","(160, 254, 97)","(152, 273, 98)",...,1.0,7.0,NaN,NaN,576460752504703640,576460752481353530,EPG_L3_54939,54939.0,NaN,NaN
72723,73151,73151,73151,73151,"(34376, 8935, 1579)","(34357, 8944, 1579)","(10, 10, 7)","(10, 10, 7)","(165, 275, 2)","(146, 284, 2)",...,1.0,7.0,NaN,NaN,576460752504703640,576460752499304326,EPG_L3_54939,54939.0,NaN,NaN
74354,74787,74787,74787,74787,"(34642, 9312, 1629)","(34668, 9324, 1628)","(10, 11, 7)","(10, 11, 7)","(431, 152, 52)","(457, 164, 51)",...,1.0,7.0,NaN,NaN,576460752504703640,576460752578156777,EPG_L3_54939,54939.0,NaN,NaN
74355,74788,74788,74788,74788,"(34638, 9316, 1630)","(34664, 9297, 1628)","(10, 11, 7)","(10, 11, 7)","(427, 156, 53)","(453, 137, 51)",...,1.0,7.0,1.0,6.0,576460752504703640,576460752521344750,EPG_L3_54939,54939.0,EPG_L2_55944,55944.0
74356,74789,74789,74789,74789,"(34644, 9312, 1630)","(34670, 9325, 1630)","(10, 11, 7)","(10, 11, 7)","(433, 152, 53)","(459, 165, 53)",...,1.0,7.0,NaN,NaN,576460752504703640,576460752578156777,EPG_L3_54939,54939.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278067,279674,279674,279674,279674,"(38882, 10083, 1548)","(38896, 10058, 1549)","(19, 12, 6)","(19, 12, 6)","(171, 423, 71)","(185, 398, 72)",...,1.0,7.0,NaN,NaN,576460752504703640,576460752545029027,EPG_L3_54939,54939.0,NaN,NaN
278076,279683,279683,279683,279683,"(38930, 10114, 1555)","(38933, 10140, 1555)","(19, 12, 6)","(19, 12, 6)","(219, 454, 78)","(222, 480, 78)",...,1.0,7.0,NaN,NaN,576460752504703640,576460752521820378,EPG_L3_54939,54939.0,NaN,NaN
278077,279684,279684,279684,279684,"(38939, 10116, 1556)","(38965, 10100, 1556)","(19, 12, 6)","(19, 12, 6)","(228, 456, 79)","(254, 440, 79)",...,1.0,7.0,NaN,NaN,576460752504703640,576460752545029027,EPG_L3_54939,54939.0,NaN,NaN
278078,279685,279685,279685,279685,"(38931, 10113, 1556)","(38949, 10130, 1556)","(19, 12, 6)","(19, 12, 6)","(220, 453, 79)","(238, 470, 79)",...,1.0,7.0,NaN,NaN,576460752504703640,576460752521972954,EPG_L3_54939,54939.0,NaN,NaN


In [30]:
# Make sure all skids are int (or convert if needed)
expected_skids = set(nametable_exploded['skid'].dropna().astype(int).unique())

# Get all skids found in the synapse table
found_skids = set(
    pd.concat([
        syntable_filt['pre_skid'].dropna().astype(int),
        syntable_filt['post_skid'].dropna().astype(int)
    ]).unique()
)

# Find any skids that are missing entirely from the synapse table
missing_skids = expected_skids - found_skids

if missing_skids:
    print(f"Warning: {len(missing_skids)} skids missing from synapse table!")
    print("Missing skids:", sorted(missing_skids))
else:
    print("✅ All skids accounted for in synapse table.")

#Missing skids: [34931, 34955, 125250, 139986]


✅ All skids accounted for in synapse table.


In [18]:
# Filter rows whose skid is in missing_skids
filtered = nametable[ nametable['skid'].isin(missing_skids) ]

print(filtered[['skid', 'name', 'root_ids']])

Empty DataFrame
Columns: [skid, name, root_ids]
Index: []


In [31]:
# save raw
syntable_filt.to_csv('./table_csv_files/megalopta_PB1_syntable_raw_19-04-26.csv', index=False)

# format synapse table

In [55]:
'''import synapse tables
remove duplicates, autapses,
update neuron names, coltype, type columns'''

EB = pd.read_csv('./table_csv_files/megalopta_EB_syntable_raw_21-Mar-26.csv')
# PB = pd.read_csv('./table_csv_files/megalopta_PB1_syntable_raw_18-04-26.csv') 
PB = pd.read_csv('./table_csv_files/    .csv')
# PB2 = pd.read_csv('./table_csv_files/megalopta_PB2_syntable_raw_09-04-26.csv') # PB2 data not included in preprint
NO = pd.read_csv('./table_csv_files/synapses_250825/NO_all_syns_feb8_with-roots_with-sk_25-08-25.csv')

# PB2["pre_skid"] = pd.to_numeric(
#     PB2["pre_neuron_name"].str.extract(r"_(\d+)$")[0],
#     errors="coerce"
# ).astype(float)

# PB2["post_skid"] = pd.to_numeric(
#     PB2["post_neuron_name"].str.extract(r"_(\d+)$")[0],
#     errors="coerce"
# ).astype(float)

# PB2['pre_root_ids'] = "000000000"
# PB2['post_root_ids'] = "000000000"

EB = EB.rename(columns={
        'pre_pt_root_id': 'pre_root_ids',
        'post_pt_root_id': 'post_root_ids',
        'pre_pt_position': 'pre_vox_coord',
        'post_pt_position': 'post_vox_coord'})
PB = PB.rename(columns={
        'pre_pt_root_id': 'pre_root_ids',
        'post_pt_root_id': 'post_root_ids',
        'pre_pt_position': 'pre_vox_coord',
        'post_pt_position': 'post_vox_coord'})

# EB = pd.read_csv('./table_csv_files/megalopta_FBEB_syntable_unformat_22-08-25_MS.csv')

df_map = {
    'EB': EB,
    'PB': PB,
    #'PB2': PB2,
    'NO': NO
}

# # for single roi...
# df_map = {'EB':EB}

for roi, df in df_map.items():
    df = df[['pre_skid', 'post_skid', 'pre_neuron_name', 'post_neuron_name', 'pre_vox_coord', 'post_vox_coord',
            'pre_root_ids', 'post_root_ids']].rename(columns={ # 'pre_L1_ids', 'post_L1_ids'
        'pre_skid': 'pre_skid',
        'post_skid': 'post_skid',
        'pre_neuron_name': 'pre_name',
        'post_neuron_name': 'post_name',
        'pre_vox_coord': 'pre_coord',
        'post_vox_coord': 'post_coord',
        'pre_root_ids': 'pre_root_ids', 
        'post_root_ids': 'post_root_ids'
        #'pre_L1_ids': 'pre_L1_ids',
        #'post_L1_ids': 'post_L1_ids'
    })
    df['roi'] = roi
    df_map[roi] = df  # update the dictionary with the cleaned version

# unpack
EB, PB, NO = df_map['EB'], df_map['PB'], df_map['NO']

bee_conn = pd.concat((EB, PB, NO))

rows_by_roi_raw = "Rows by ROI (raw files):", {k: len(v) for k, v in df_map.items()}
print(rows_by_roi_raw)
print("Concat rows:", len(bee_conn))
# single roi
# bee_conn = df_map['EB']

bee_conn = bee_conn[bee_conn['pre_skid'] != bee_conn['post_skid']] #remove autapses

print("After autapse removal:", len(bee_conn))

bee_count = bee_conn[['pre_skid', 'post_skid', 'pre_name', 'post_name', 'pre_coord', 'post_coord', 'roi',
                     'pre_root_ids', 'post_root_ids']] #  'pre_L1_ids', 'post_L1_ids'

update_names = {'TN_LNO1_R_SH_AT': 'mLNOa_47215', # need to update
                'TN_LNO2_R_SH_AT': 'mLNOc_47221',
                'TN/LNO3_todo_AT_R_SH':	'mLNOe_56999',
                'TN/LNO4_todo_AT_R_SH':	'sLNOb_57012',
                'TN/LNO5_todo_AT_R_SH':	'mLNOd_57022',
                'TN/LNO6_todo_AT_R_SH':	'sLNOa_57028',
                'TN/LNO7_todo_AT_SH':	'mLNOf_57076',
                'TN/LNO8_todo_AT_SH':	'sLNOd_57081',
                'TN/LNO9_todo_AT_R_SH':	'sLNOc_57628'
                }


# update both pre_name and post_name where they exactly match a key
bee_count[['pre_name', 'post_name']] = (
    bee_count[['pre_name', 'post_name']].replace(update_names, regex=False)
)

bee_count["pre_skid"] = pd.to_numeric(bee_count["pre_skid"], errors="coerce").astype("Int64")
bee_count["post_skid"] = pd.to_numeric(bee_count["post_skid"], errors="coerce").astype("Int64")

C:\Users\Marcel\AppData\Local\Temp\ipykernel_5112\1040954172.py:5: DtypeWarning: Columns (16,18) have mixed types. Specify dtype option on import or set low_memory=False.
  EB = pd.read_csv('./table_csv_files/megalopta_EB_syntable_raw_21-Mar-26.csv')


KeyError: "['pre_neuron_name', 'post_neuron_name', 'pre_vox_coord', 'post_vox_coord'] not in index"

In [33]:
# ensure there are no duplicates
# rows where (pre_coord, post_coord) appear more than once
dups = bee_count[bee_count.duplicated(subset=["pre_coord", "post_coord"], keep=False)]
dups

,pre_skid,post_skid,pre_name,post_name,pre_coord,post_coord,roi,pre_root_ids,post_root_ids
93,<NA>,<NA>,NaN,NaN,"(19315, 15173, 1531)","(19313, 15186, 1531)",EB,576460753009221950,576460752521302041
94,<NA>,<NA>,NaN,NaN,"(19315, 15173, 1531)","(19313, 15186, 1531)",EB,576460753009221950,576460752521302041
101,<NA>,<NA>,NaN,NaN,"(19315, 15173, 1531)","(19303, 15187, 1532)",EB,576460753009221950,576460752521346585
102,<NA>,<NA>,NaN,NaN,"(19315, 15173, 1531)","(19303, 15187, 1532)",EB,576460753009221950,576460752521346585
657,<NA>,<NA>,NaN,NaN,"(19198, 15724, 1695)","(19186, 15746, 1694)",EB,576460753027942666,576460753092156252
...,...,...,...,...,...,...,...,...,...
570870,<NA>,<NA>,NaN,NaN,"(45501, 9703, 1170)","(45495, 9681, 1170)",PB,576460752530342918,576460752536763372
570871,<NA>,<NA>,NaN,NaN,"(45501, 9703, 1170)","(45513, 9678, 1171)",PB,576460752530342918,576460752533749228
570872,<NA>,<NA>,NaN,NaN,"(45501, 9703, 1170)","(45513, 9678, 1171)",PB,576460752530342918,576460752533749228
570873,<NA>,<NA>,NaN,NaN,"(45501, 9703, 1170)","(45480, 9689, 1171)",PB,576460752530342918,576460752501201611


In [34]:
# Identify groups that have duplicates
dup_mask = bee_count.duplicated(subset=["pre_coord", "post_coord"], keep=False)

# For each duplicated group, drop exactly one row
to_drop = bee_count[dup_mask].groupby(["pre_coord", "post_coord"]).head(1).index

# New DF with one instance removed
bee_count = bee_count.drop(index=to_drop)


In [35]:
rows_by_roi_raw = "Rows by ROI (raw files):", {k: len(v) for k, v in df_map.items()}
print(rows_by_roi_raw)

('Rows by ROI (raw files):', {'EB': 7001133, 'PB': 673740, 'NO': 198118})


In [36]:
# add brain side (hemisphere) for each pre and post neuron if data available

def get_side(name):
    if pd.isna(name):
        return np.nan

    # rule 1: explicit _L_ or _R_
    if "_L_" in name:
        return "left"
    if "_R_" in name:
        return "right"

    # rule 2: _L#_ or _R#_ but NOT mixed like L4R5
    # Find all matches like L3, R2 that are surrounded by underscores
    matches = re.findall(r'_(L\d+|R\d+)_', name)

    if len(matches) == 1:
        if matches[0].startswith("L"):
            return "left"
        elif matches[0].startswith("R"):
            return "right"

    # if multiple matches (e.g., L4R5) → ambiguous → ignore
    return np.nan


# Apply to dataframe
bee_count["pre_side"] = bee_count["pre_name"].apply(get_side)
bee_count["post_side"] = bee_count["post_name"].apply(get_side)

In [37]:
''' fix names, add columns for neuron type_col (e.g. EPG_L2) and type (e.g EPG)'''

pattern_standard = r'^([A-Z]{3}[a-zA-Z]?(?:\d+)?(?:_[a-z])?).*(R\d|L\d)'
pattern_er = r'(ER|ExR).*(L|R).*(MBUv|MBUd|LBU|LAL|ExR)'
pattern_d7 = r'(?i)(delta7).*?([LR]\d)[ _-]?([LR]\d)'
pattern_lno = r'([a-zA-Z])LNO([a-zA-Z]?)'

extracted_pre_standard = bee_count['pre_name'].str.extract(pattern_standard)
extracted_post_standard = bee_count['post_name'].str.extract(pattern_standard)

extracted_pre_er = bee_count['pre_name'].str.extract(pattern_er)
extracted_post_er = bee_count['post_name'].str.extract(pattern_er)

extracted_pre_d7 = bee_count['pre_name'].str.extract(pattern_d7)
extracted_post_d7 = bee_count['post_name'].str.extract(pattern_d7)

extracted_pre_lno = bee_count['pre_name'].str.extract(pattern_lno)
extracted_post_lno = bee_count['post_name'].str.extract(pattern_lno)

# precompute the fully constructed strings for each pattern
d7_type_pre = extracted_pre_d7[0] + '_' + extracted_pre_d7[1] + extracted_pre_d7[2]
d7_type_post = extracted_post_d7[0] + '_' + extracted_post_d7[1] + extracted_post_d7[2]

er_type_pre = extracted_pre_er[0] + '_' + extracted_pre_er[1] + '_' + extracted_pre_er[2]
er_type_post = extracted_post_er[0] + '_' + extracted_post_er[1] + '_' + extracted_post_er[2]

std_type_pre = extracted_pre_standard[0] + '_' + extracted_pre_standard[1]
std_type_post = extracted_post_standard[0] + '_' + extracted_post_standard[1]

lno_type_pre= extracted_pre_lno[0] + 'LNO' + '_' + extracted_pre_lno[1]
lno_type_post = extracted_post_lno[0] + 'LNO' + '_' + extracted_post_lno[1]

# combine them
bee_count['type_pre_col'] = er_type_pre.combine_first(std_type_pre).combine_first(d7_type_pre).combine_first(lno_type_pre)
bee_count['type_post_col'] = er_type_post.combine_first(std_type_post).combine_first(d7_type_post).combine_first(lno_type_post)

# add type columns
bee_count['type_pre'] = extracted_pre_er[2].combine_first(extracted_pre_standard[0]).combine_first(extracted_pre_d7[0]).combine_first(lno_type_pre)
bee_count['type_post'] = extracted_post_er[2].combine_first(extracted_post_standard[0]).combine_first(extracted_post_d7[0]).combine_first(lno_type_post)

# final naming
bee_count['pre_name'] = bee_count['type_pre_col'] + '_' + bee_count['pre_skid'].astype(str)
bee_count['post_name'] = bee_count['type_post_col'] + '_' + bee_count['post_skid'].astype(str)

bee_count

,pre_skid,post_skid,pre_name,post_name,pre_coord,post_coord,roi,pre_root_ids,post_root_ids,pre_side,post_side,type_pre_col,type_post_col,type_pre,type_post
0,<NA>,<NA>,NaN,NaN,"(19151, 15985, 1290)","(19126, 15985, 1291)",EB,576460753050011843,576460753050011843,NaN,NaN,NaN,NaN,NaN,NaN
1,<NA>,<NA>,NaN,NaN,"(19166, 15985, 1295)","(19150, 15980, 1297)",EB,576460753050011843,576460753050011843,NaN,NaN,NaN,NaN,NaN,NaN
2,<NA>,<NA>,NaN,NaN,"(19151, 15794, 1303)","(19160, 15816, 1303)",EB,576460753121451657,576460753121451657,NaN,NaN,NaN,NaN,NaN,NaN
3,<NA>,<NA>,NaN,NaN,"(19161, 15792, 1302)","(19138, 15808, 1302)",EB,576460753121451657,576460753121451657,NaN,NaN,NaN,NaN,NaN,NaN
4,<NA>,<NA>,NaN,NaN,"(19149, 15787, 1303)","(19163, 15797, 1303)",EB,576460753121451657,576460753121451657,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198113,<NA>,<NA>,NaN,NaN,"(29675, 17287, 4937)","(29663, 17313, 4936)",NO,504403158273317883,504403158304062595,NaN,NaN,NaN,NaN,NaN,NaN
198114,<NA>,<NA>,NaN,NaN,"(29663, 17450, 4940)","(29655, 17433, 4939)",NO,504403158303005628,504403158299907380,NaN,NaN,NaN,NaN,NaN,NaN
198115,<NA>,<NA>,NaN,NaN,"(29665, 16818, 5000)","(29689, 16821, 5000)",NO,504403158424141988,504403158359497680,NaN,NaN,NaN,NaN,NaN,NaN
198116,<NA>,<NA>,NaN,NaN,"(29826, 16886, 5034)","(29814, 16865, 5033)",NO,504403158292205026,504403158296470108,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
# fixing messy naming and annotations cont...

bee_count = bee_count.replace({'PEG_L1': 'PEG_L1R1'}, regex=True)
bee_count = bee_count.replace({'EPG_L1': 'EPG_L1R1'}, regex=True)

name_fixes = {
    "L2L1": "L2L10R7",
    "L7R2": "L7R2R10",
    "L8R1": "L8R1R9",
    "L1L9": "L1L9R8",
}

for old, new in name_fixes.items():
    pattern = rf'delta7_{old}.*'
    replacement = f'delta7_{new}'
    for col in ['type_pre_col', 'type_post_col']:
        bee_count[col] = bee_count[col].str.replace(pattern, replacement, regex=True)

# Fix full neuron name columns
for old, new in name_fixes.items():
    pattern = rf'^(delta7_)({old})(_.+)$'
    replacement = rf'\1{new}\3'
    for col in ['pre_name', 'post_name']:
        bee_count[col] = bee_count[col].str.replace(pattern, replacement, regex=True)

bee_count = bee_count.apply( # upper case "d" to match fly nomenclature
    lambda col: col.str.replace("delta7", "Delta7", regex=False)
    if col.dtype == "object" else col
)


In [39]:
# verify type level counts are still in expected range
bee_pb = bee_count[bee_count['roi'].str.contains(r'PB')]
bee_eb = bee_count[bee_count['roi'].str.contains(r'EB')] 
bee_no = bee_count[bee_count['roi'].str.contains(r'NO')] 
roimap = [bee_pb, bee_eb, bee_no]
print(rows_by_roi_raw)
print("Rows by ROI (names updated):", {len(k) for k in roimap})

('Rows by ROI (raw files):', {'EB': 7001133, 'PB': 673740, 'NO': 198118})
Rows by ROI (names updated): {197255, 671066, 6971679}


In [45]:
bee_count[bee_count['pre_name'].str.contains(r'Delta7_L1', na=False)]

,pre_skid,post_skid,pre_name,post_name,pre_coord,post_coord,roi,pre_root_ids,post_root_ids,pre_side,post_side,type_pre_col,type_post_col,type_pre,type_post
19733,78122,<NA>,Delta7_L1L9R8_78122,NaN,"(36196, 6354, 1225)","(36222, 6356, 1223)",PB,576460752485062902,576460752502116862,left,NaN,Delta7_L1L9R8,NaN,Delta7,NaN
23906,82977,<NA>,Delta7_L1L9R8_82977,NaN,"(33682, 6684, 1429)","(33689, 6670, 1432)",PB,576460752513584969,576460752326513571,left,NaN,Delta7_L1L9R8,NaN,Delta7,NaN
42303,81004,<NA>,Delta7_L1L9R8_81004,NaN,"(34270, 6990, 1450)","(34278, 6965, 1451)",PB,576460752545395043,576460752426196064,left,NaN,Delta7_L1L9R8,NaN,Delta7,NaN
45648,76634,<NA>,Delta7_L1L9R8_76634,NaN,"(35147, 6545, 1358)","(35122, 6520, 1359)",PB,576460752542700737,576460752420188590,left,NaN,Delta7_L1L9R8,NaN,Delta7,NaN
48025,81004,112687,Delta7_L1L9R8_81004,PEN_b_L2_112687,"(35049, 7271, 1417)","(35024, 7281, 1417)",PB,576460752545395043,576460752507193318,left,left,Delta7_L1L9R8,PEN_b_L2,Delta7,PEN_b
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
644111,84275,<NA>,Delta7_L1L9R8_84275,NaN,"(48836, 9517, 1755)","(48811, 9512, 1755)",PB,576460752525921670,576460752427321906,left,NaN,Delta7_L1L9R8,NaN,Delta7,NaN
646307,81004,<NA>,Delta7_L1L9R8_81004,NaN,"(49556, 8275, 1757)","(49557, 8255, 1757)",PB,576460752545395043,576460752411422808,left,NaN,Delta7_L1L9R8,NaN,Delta7,NaN
653824,81004,<NA>,Delta7_L1L9R8_81004,NaN,"(49967, 7061, 1830)","(49993, 7067, 1831)",PB,576460752545395043,576460752395034469,left,NaN,Delta7_L1L9R8,NaN,Delta7,NaN
669547,81014,<NA>,Delta7_L1L9R8_81014,NaN,"(47713, 7852, 1955)","(47688, 7861, 1955)",PB,576460752581659199,576460752430423400,left,NaN,Delta7_L1L9R8,NaN,Delta7,NaN


In [40]:
#############################################
################ EXPORT #####################
#############################################
bee_count.to_csv('./table_csv_files/megalopta_cx_syntable_19-04-26.csv', index=False)

# make connectivity table

In [52]:
bee_conn = pd.read_csv('../syntables/table_csv_files/megalopta_cx_syntable_20-04-26.csv')
# bee_conn = bee_count.copy()

bee_conn = bee_conn[bee_conn['pre_skid'] != bee_conn['post_skid']] #remove autapses (already did above, just to be sure)

df = bee_conn[['pre_skid', 'post_skid', 'pre_name', 'post_name', 'type_pre_col', 'type_post_col', 
               'type_pre', 'type_post', 'roi', 'pre_side', 'post_side']]

'''keep NaN partners so that relative weight score is calculated with total synapse numbers 
makes it comparable to the fruit fly value_counts() here counts how many times each unique 
combination of the 4 columns occurs in the df'''
contable = df.value_counts(dropna=False)

# reset index turns MultiIndex into columns
contable = contable.reset_index().rename(
    columns={
        0: 'count'  # This is the name of the column created by value_counts()
    }
)

In [53]:
d = bee_conn[bee_conn['type_pre_col'].str.contains(r'Delta7_L1', na=False)]
d['type_pre_col'].unique()

array(['Delta7_L1L9R8'], dtype=object)

In [54]:
#############################################
################ EXPORT #####################
#############################################

contable.to_csv('./table_csv_files/megalopta_cx_conntable_20-04-26.csv', index=False)

### calculate relative weights per ROI (NOTE: not used in preprint)

In [37]:
'''add relative weight column and apply synapse count threshold: (NOTE: this isn't currently used in analysis)

Take synapses from neuron a to neuron b in region X and divided this 
number by the total number of inputs that neuron b received in ROI X

probably expects skid to be float
'''

# Replace 'nan' (the string)
bee_count['pre_skid'] = bee_count['pre_skid'].fillna(0.0)
bee_count['post_skid'] = bee_count['post_skid'].fillna(0.0)
bee_count['pre_name'] = bee_count['pre_name'].fillna('unknown')
bee_count['post_name'] = bee_count['post_name'].fillna('unknown')

# add column for relative weights: take synapses from neuron a to neuron b in region X and divided this 
# number by the total number of inputs that neuron b received in X

bee_count['pre_skid'] = bee_count['pre_skid'].astype(float).astype(int)
bee_count['post_skid'] = bee_count['post_skid'].astype(float).astype(int)

# Group by both 'post_skid' and 'roi', then sum the synapse count
total_post_syn_by_roi = bee_count.groupby(['post_skid', 'roi'])['count'].sum().reset_index(name='total_post_syn')
# Merge this total count back to the original DataFrame
bee_count = pd.merge(bee_count, total_post_syn_by_roi, on=['post_skid', 'roi'])

bee_count['pre_skid'] = bee_count['pre_skid'].astype(float).astype(int)
bee_count['post_skid'] = bee_count['post_skid'].astype(float).astype(int)

# Normalize the synaptic counts
bee_count['relative_weight'] = bee_count['count'] / bee_count['total_post_syn']

# remove any unknown partners
bee_count = bee_count[(bee_count['pre_skid'] != 0) & (bee_count['post_skid'] != 0)]
bee_count = bee_count[~(bee_count['pre_name'].str.contains(r'unknown') | bee_count['post_name'].str.contains(r'unknown'))]

'''NOTE THIS THRESHOLD WAS REMOVED TO INCLUDE ER_LAL NEURONS, ALL OF WHICH HAVE FEWER THAN FIVE CONNECTIONS'''
# remove connections below 5 synapses (same as in Hulse et al.)
# bee_count = bee_count[bee_count['count']>5]

bee_count

,pre_skid,post_skid,pre_name,post_name,roi,pre_side,post_side,count,total_post_syn,relative_weight
485,56351,56356,PEG_R8_56351.0,EPG_R8_56356.0,EB,right,right,76,5734,0.013254
486,63772,56356,PEN_b_L3_63772.0,EPG_R8_56356.0,EB,left,right,62,5734,0.010813
487,108339,56356,PEN_a_R9_108339.0,EPG_R8_56356.0,EB,right,right,58,5734,0.010115
488,64257,56356,PEN_b_R9_64257.0,EPG_R8_56356.0,EB,right,right,41,5734,0.007150
489,112713,56356,PEN_a_L2_112713.0,EPG_R8_56356.0,EB,left,right,41,5734,0.007150
...,...,...,...,...,...,...,...,...,...,...
9307,57076,264448,mLNO_f_57076.0,PFN_L2_264448.0,NO,NaN,left,3,5,0.600000
9332,57081,34212,sLNO_d_57081.0,PEN_L6_34212.0,NO,NaN,left,1,2,0.500000
9336,154881,184797,ER_R_MBUv_154881.0,ER_R_MBUv_184797.0,NO,right,right,1,1,1.000000
9346,275602,117271,PFN_L2_275602.0,PFN_L6_117271.0,NO,left,left,1,3,0.333333


In [ ]:
# already done while formatting syntable

# # Regex patterns
# pattern_standard = r'^([A-Z]{3}[a-zA-Z]?(?:\d+)?(?:_[a-z])?).*(R\d|L\d)'
# pattern_er = r'(ER|ExR).*(L|R).*(MBUv|MBUd|LBU|LAL|ExR)'
# pattern_d7 = r'(?i)(delta7).*?([LR]\d)[ _-]?([LR]\d)'
# pattern_lno = r'([a-zA-Z])LNO([a-zA-Z]?)'

# # Extract for standard neurons
# extracted_pre_standard = bee_count['pre_name'].str.extract(pattern_standard)
# extracted_post_standard = bee_count['post_name'].str.extract(pattern_standard)

# # Extract for er neurons
# extracted_pre_er = bee_count['pre_name'].str.extract(pattern_er)
# extracted_post_er = bee_count['post_name'].str.extract(pattern_er)

# # Extract for d7 neurons
# extracted_pre_d7 = bee_count['pre_name'].str.extract(pattern_d7)
# extracted_post_d7 = bee_count['post_name'].str.extract(pattern_d7)

# # Extract for lno neurons
# extracted_pre_lno = bee_count['pre_name'].str.extract(pattern_lno)
# extracted_post_lno = bee_count['post_name'].str.extract(pattern_lno)

# # precompute the fully constructed strings for each pattern
# d7_type_pre = extracted_pre_d7[0] + '_' + extracted_pre_d7[1] + extracted_pre_d7[2]
# d7_type_post = extracted_post_d7[0] + '_' + extracted_post_d7[1] + extracted_post_d7[2]

# er_type_pre = extracted_pre_er[0] + '_' + extracted_pre_er[1] + '_' + extracted_pre_er[2]
# er_type_post = extracted_post_er[0] + '_' + extracted_post_er[1] + '_' + extracted_post_er[2]

# std_type_pre = extracted_pre_standard[0] + '_' + extracted_pre_standard[1]
# std_type_post = extracted_post_standard[0] + '_' + extracted_post_standard[1]

# lno_type_pre= extracted_pre_lno[0] + 'LNO' + '_' + extracted_pre_lno[1]
# lno_type_post = extracted_post_lno[0] + 'LNO' + '_' + extracted_post_lno[1]

# # safely combine them
# bee_count['type_pre_col'] = er_type_pre.combine_first(std_type_pre).combine_first(d7_type_pre).combine_first(lno_type_pre)
# bee_count['type_post_col'] = er_type_post.combine_first(std_type_post).combine_first(d7_type_post).combine_first(lno_type_post)

# # assign type (e.g., only the class, not location info)
# bee_count['type_pre'] = extracted_pre_er[2].combine_first(extracted_pre_standard[0]).combine_first(extracted_pre_d7[0]).combine_first(lno_type_pre)
# bee_count['type_post'] = extracted_post_er[2].combine_first(extracted_post_standard[0]).combine_first(extracted_post_d7[0]).combine_first(lno_type_post)

# # final naming
# bee_count['pre_name'] = bee_count['type_pre_col'] + '_' + bee_count['pre_skid'].astype(str)
# bee_count['post_name'] = bee_count['type_post_col'] + '_' + bee_count['post_skid'].astype(str)

# bee_count

In [29]:
# Give relative weights their own column based on ROI 
bee_count['relative_weight_PB'] = 0.0
bee_count['relative_weight_EB'] = 0.0
bee_count['relative_weight_NO'] = 0.0

bee_count.loc[bee_count['roi'] == 'PB', 'relative_weight_PB'] = bee_count['relative_weight']
bee_count.loc[bee_count['roi'] == 'EB', 'relative_weight_EB'] = bee_count['relative_weight']
bee_count.loc[bee_count['roi'] == 'NO', 'relative_weight_NO'] = bee_count['relative_weight']

bee_count = bee_count.drop(columns=['relative_weight'])

bee_count['PB_post'] = 0.0
bee_count['EB_post'] = 0.0
bee_count['NO_post'] = 0.0

bee_count.loc[bee_count['roi'] == 'PB', 'PB_post'] = bee_count['total_post_syn']
bee_count.loc[bee_count['roi'] == 'EB', 'EB_post'] = bee_count['total_post_syn']
bee_count.loc[bee_count['roi'] == 'NO', 'NO_post'] = bee_count['total_post_syn']

bee_count = bee_count.drop(columns=['total_post_syn'])

bee_count

,pre_skid,post_skid,pre_name,post_name,roi,roi_main,count,type_pre_col,type_post_col,type_pre,type_post,relative_weight_PB,relative_weight_EB,relative_weight_NO,PB_post,EB_post,NO_post
529,56351,56356,PEG_R8_56351,EPG_R8_56356,EB,EB,76,PEG_R8,EPG_R8,PEG,EPG,0.0,0.013254,0.000000,0.0,5734.0,0.0
530,63772,56356,PEN_b_L3_63772,EPG_R8_56356,EB,EB,62,PEN_b_L3,EPG_R8,PEN_b,EPG,0.0,0.010813,0.000000,0.0,5734.0,0.0
531,108339,56356,PEN_a_R9_108339,EPG_R8_56356,EB,EB,58,PEN_a_R9,EPG_R8,PEN_a,EPG,0.0,0.010115,0.000000,0.0,5734.0,0.0
532,64257,56356,PEN_b_R9_64257,EPG_R8_56356,EB,EB,41,PEN_b_R9,EPG_R8,PEN_b,EPG,0.0,0.007150,0.000000,0.0,5734.0,0.0
533,112713,56356,PEN_a_L2_112713,EPG_R8_56356,EB,EB,41,PEN_a_L2,EPG_R8,PEN_a,EPG,0.0,0.007150,0.000000,0.0,5734.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9640,57076,264448,mLNO__57076,PFN_L2_264448,NO,NO,3,mLNO_,PFN_L2,mLNO_,PFN,0.0,0.000000,0.600000,0.0,0.0,5.0
9688,57081,34212,sLNO__57081,PEN_L6_34212,NO,NO,1,sLNO_,PEN_L6,sLNO_,PEN,0.0,0.000000,0.500000,0.0,0.0,2.0
9696,275602,117271,PFN_L2_275602,PFN_L6_117271,NO,NO,1,PFN_L2,PFN_L6,PFN,PFN,0.0,0.000000,0.333333,0.0,0.0,3.0
9699,154881,184797,ER_R_MBUv_154881,ER_R_MBUv_184797,NO,NO,1,ER_R_MBUv,ER_R_MBUv,MBUv,MBUv,0.0,0.000000,1.000000,0.0,0.0,1.0


In [30]:
# bee_count = bee_count.replace({'delta7_L2R7': 'delta7_L2L10R7', 'delta7_L7R2': 'delta7_L7R2R10'}, regex=True)
# bee_count = bee_count.replace({'delta7_L9L1': 'delta7_L9L1R8'}, regex=True)
# bee_count = bee_count.replace({'delta7_L8R1': 'Delta7_L8R1R9'}, regex=True)
bee_count = bee_count.replace({'PEG_R1': 'PEG_L_R1L1'}, regex=True)
bee_count = bee_count.replace({'EPG_L1': 'EPG_L_R1L1'}, regex=True)
bee_count = bee_count.replace({'delta7': 'Delta7'}, regex=True)

In [31]:
bee_count[bee_count['pre_name'].str.contains(r'Delta7_L8', na=False)]

,pre_skid,post_skid,pre_name,post_name,roi,roi_main,count,type_pre_col,type_post_col,type_pre,type_post,relative_weight_PB,relative_weight_EB,relative_weight_NO,PB_post,EB_post,NO_post
1013,84265,42175,Delta7_L8R1R9_84265,PEG_L5_42175,PB,PB,1,Delta7_L8R1R9,PEG_L5,Delta7,PEG,0.000266,0.0,0.0,3753.0,0.0,0.0
1014,83758,42175,Delta7_L8R1R9_83758,PEG_L5_42175,PB,PB,1,Delta7_L8R1R9,PEG_L5,Delta7,PEG,0.000266,0.0,0.0,3753.0,0.0,0.0
1318,81019,35102,Delta7_L8R1R9_81019,PEN_b_L5_35102,PB,PB,1,Delta7_L8R1R9,PEN_b_L5,Delta7,PEN_b,0.000257,0.0,0.0,3895.0,0.0,0.0
1369,84265,99063,Delta7_L8R1R9_84265,PEG_L3_99063,PB,PB,3,Delta7_L8R1R9,PEG_L3,Delta7,PEG,0.001047,0.0,0.0,2866.0,0.0,0.0
1371,83698,99063,Delta7_L8R1R9_83698,PEG_L3_99063,PB,PB,2,Delta7_L8R1R9,PEG_L3,Delta7,PEG,0.000698,0.0,0.0,2866.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9231,84265,84466,Delta7_L8R1R9_84265,Delta7_L5R4_84466,PB2,PB,2,Delta7_L8R1R9,Delta7_L5R4,Delta7,Delta7,0.000000,0.0,0.0,0.0,0.0,0.0
9232,81019,84466,Delta7_L8R1R9_81019,Delta7_L5R4_84466,PB2,PB,2,Delta7_L8R1R9,Delta7_L5R4,Delta7,Delta7,0.000000,0.0,0.0,0.0,0.0,0.0
9235,83758,84466,Delta7_L8R1R9_83758,Delta7_L5R4_84466,PB2,PB,1,Delta7_L8R1R9,Delta7_L5R4,Delta7,Delta7,0.000000,0.0,0.0,0.0,0.0,0.0
9386,81019,42181,Delta7_L8R1R9_81019,Delta7_L2L1_42181,PB2,PB,2,Delta7_L8R1R9,Delta7_L2L1,Delta7,Delta7,0.000000,0.0,0.0,0.0,0.0,0.0


In [32]:
#############################################
################ EXPORT #####################
#############################################

bee_count.to_csv('./table_csv_files/megalopta_cx_conntable_09-04-26.csv', index=False)

----